In [0]:
from pyspark.sql.functions import col, sum, avg, count, min, max, year, month, dayofmonth, dayofweek, quarter, weekofyear, date_format, to_date, monotonically_increasing_id, when, round, upper
from pyspark.sql.window import Window

# create separate schemas for dimensions and facts
spark.sql("CREATE SCHEMA IF NOT EXISTS dim")
spark.sql("CREATE SCHEMA IF NOT EXISTS fact")

print("Schemas created: dim and fact")

In [0]:
# DIMENSION TABLES 

#  dim_customers 
dim_customers = (
    spark.table("silver.customers")
    .select(
        col("customer_id").alias("customer_key"),
        col("customer_id"),
        col("customer_name"),
        col("customer_type"),
        upper(col("country")).alias("country"),
        col("signup_date"),
        col("ingestion_ts")
    )
)

dim_customers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim.dim_customers")
print(f"dim_customers created: {dim_customers.count()} records")
display(dim_customers.limit(5))

# dim_products
dim_products = (
    spark.table("silver.products")
    .select(
        col("product_id").alias("product_key"),
        col("product_id"),
        col("product_name"),
        col("category"),
        col("ingestion_ts")
    )
)

dim_products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim.dim_products")
print(f"dim_products created: {dim_products.count()} records")
display(dim_products.limit(5))

# dim_regions
dim_regions = (
    spark.table("silver.regions")
    .select(
        col("region_code").alias("region_key"),
        col("region_code"),
        col("region_name"),
        upper(col("country")).alias("country"),
        col("ingestion_ts")
    )
)

dim_regions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim.dim_regions")
print(f"dim_regions created: {dim_regions.count()} records")
display(dim_regions.limit(5))

In [0]:
# dim_exchange_rates 

silver_exchange_rates = spark.table("silver.exchange_rates")

dim_exchange_rates = (
    silver_exchange_rates
    .select(
        col("currency"),
        col("date"),
        col("rate_to_usd"),
        col("ingestion_ts")
    )
)

dim_exchange_rates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim.dim_exchange_rates")
print(f"dim_exchange_rates created: {dim_exchange_rates.count()} records")

print("\nSample from dim_exchange_rates:")
display(dim_exchange_rates.limit(5))

In [0]:
# dim_dates 

silver_invoices = spark.table("silver.invoices")
silver_payments = spark.table("silver.payments")

invoice_dates = silver_invoices.select(col("invoice_date").alias("date"))
payment_dates = silver_payments.select(col("payment_date").alias("date"))

all_dates = invoice_dates.union(payment_dates).distinct().filter(col("date").isNotNull())

dim_dates = (
    all_dates
    .select(
        col("date").alias("date_key"),
        col("date"),
        year(col("date")).alias("year"),
        quarter(col("date")).alias("quarter"),
        month(col("date")).alias("month"),
        date_format(col("date"), "MMMM").alias("month_name"),
        weekofyear(col("date")).alias("week_of_year"),
        dayofmonth(col("date")).alias("day_of_month"),
        dayofweek(col("date")).alias("day_of_week"),
        date_format(col("date"), "EEEE").alias("day_name")
    )
    .orderBy("date")
)

dim_dates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim.dim_dates")
print(f"dim_dates created: {dim_dates.count()} records")

# Display sample
print("\nSample from dim_dates:")
display(dim_dates.limit(5))

In [0]:
# FACT TABLES 

#  fact_invoices
silver_invoices = spark.table("silver.invoices")
silver_invoice_line_items = spark.table("silver.invoice_line_items")

invoice_metrics = (
    spark.table("silver.invoice_line_items")
    .groupBy("invoice_id")
    .agg(
        sum("line_total").alias("total_amount"),
        sum("quantity").alias("total_quantity"),
        count("invoice_line_id").alias("line_item_count"),
        avg("discount").alias("avg_discount")
    )
)

fact_invoices = (
    spark.table("silver.invoices")
    .alias("inv")
    .join(
        invoice_metrics.alias("metrics"),
        col("inv.invoice_id") == col("metrics.invoice_id"),
        "left"
    )
    .select(
        col("inv.invoice_id"),
        col("inv.customer_id").alias("customer_key"),
        col("inv.invoice_date").alias("date_key"),
        col("inv.region_code").alias("region_key"),
        col("inv.currency"),
        col("inv.invoice_status"),
        col("metrics.total_amount"),
        col("metrics.total_quantity"),
        col("metrics.line_item_count"),
        col("metrics.avg_discount"),
        col("inv.ingestion_ts")
    )
)

fact_invoices.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact.fact_invoices")
print(f"fact_invoices created: {fact_invoices.count()} records")
display(fact_invoices.limit(5))

In [0]:
# fact_invoice_line_items

silver_invoice_line_items = spark.table("silver.invoice_line_items")

fact_invoice_line_items = (
    silver_invoice_line_items
    .select(
        col("invoice_line_id"),
        col("invoice_id"),
        col("customer_id").alias("customer_key"),
        col("product_id").alias("product_key"),
        col("invoice_date").alias("date_key"),
        col("quantity"),
        col("unit_price"),
        col("discount"),
        col("line_total"),
        col("cost_price"),
        (col("line_total") - (col("quantity") * col("cost_price"))).alias("profit"),
        col("ingestion_ts")
    )
)

fact_invoice_line_items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact.fact_invoice_line_items")
print(f"fact_invoice_line_items created: {fact_invoice_line_items.count()} records")
display(fact_invoice_line_items.limit(5))

In [0]:
# fact_payments

silver_payments = spark.table("silver.payments")

fact_payments = (
    silver_payments
    .select(
        col("payment_id"),
        col("invoice_id"),
        col("customer_id").alias("customer_key"),
        col("payment_date").alias("date_key"),
        col("payment_amount"),
        col("payment_method"),
        col("invoice_currency"),
        col("invoice_date"),
        col("ingestion_ts")
    )
)

fact_payments.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact.fact_payments")
print(f"fact_payments created: {fact_payments.count()} records")
display(fact_payments.limit(5))

In [0]:
# VALIDATION

print("\n" + "="*60)
print("DIMENSIONAL MODEL CREATED SUCCESSFULLY")
print("="*60)

print("\n DIMENSION TABLES (dim schema) ")
display(spark.sql("SHOW TABLES IN dim"))

print("\nFACT TABLES (fact schema)")
display(spark.sql("SHOW TABLES IN fact"))

print("\nRECORD COUNTS")
print(f"dim_customers: {spark.table('dim.dim_customers').count()}")
print(f"dim_products: {spark.table('dim.dim_products').count()}")
print(f"dim_regions: {spark.table('dim.dim_regions').count()}")
print(f"dim_exchange_rates: {spark.table('dim.dim_exchange_rates').count()}")
print(f"dim_dates: {spark.table('dim.dim_dates').count()}")
print(f"fact_invoices: {spark.table('fact.fact_invoices').count()}")
print(f"fact_invoice_line_items: {spark.table('fact.fact_invoice_line_items').count()}")
print(f"fact_payments: {spark.table('fact.fact_payments').count()}")

print("\n dimensional and fact tables created ")